# Traffic-Sign-Classification — Colab 실행

ANN / DNN / CNN 구조 비교 (Keras + PyTorch, 둘 다 GPU).

**준비:** `crop.zip` (상위 10개 표지판 코드 × 1,000장)을 Google Drive 루트에 올려둔다.

메뉴 → 런타임 유형 → **T4 GPU** 선택 후 위에서부터 순서대로 실행.

In [ ]:
# 1. 환경 확인
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
import tensorflow as tf
print('tf', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2. 저장소 clone
!git clone https://github.com/seo941024/Traffic-Sign-Classification.git
%cd Traffic-Sign-Classification

In [ ]:
# 3. crop 데이터 가져오기 (Drive에 올려둔 crop.zip 경로를 맞게 수정)
from google.colab import drive
drive.mount('/content/drive')

CROP_ZIP = '/content/drive/MyDrive/crop.zip'   # ← 본인 경로로 수정

import zipfile, os
os.makedirs('crop', exist_ok=True)
with zipfile.ZipFile(CROP_ZIP) as z:
    z.extractall('crop')
classes = sorted(d for d in os.listdir('crop') if os.path.isdir(f'crop/{d}'))
print(len(classes), '클래스:', classes)
for c in classes:
    print(f'  {c}: {len(os.listdir(f"crop/{c}")):,}장')

In [ ]:
# 3-1. config.py 의 CLASS_NAMES 가 crop 폴더와 일치하는지 확인
import config
assert sorted(config.CLASS_NAMES) == classes, (config.CLASS_NAMES, classes)
print('OK:', config.CLASS_NAMES)

In [ ]:
# 4. PyTorch — ANN / DNN / CNN
!python train_torch.py --epochs 25

In [ ]:
# 5. Keras — ANN / DNN / CNN (Colab 에선 GPU)
!python train_keras.py --epochs 25

## 6. 결과 분석 — 혼동행렬 + 오분류 샘플

기본 스크립트는 정확도만 출력하므로, 어떤 표지판을 서로 헷갈리는지를 따로 본다.
213↔214 (좌/우회전금지), 308↔309 (화살표 방향) 처럼 모양만 다른 쌍이 ANN 에서 무너지는지 보면
"Flatten 하면 공간정보를 잃는다"는 주장의 직접 증거가 된다.

In [ ]:
import torch, numpy as np
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import config
from src.torch_impl.models import build_model

SEED = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tf_ = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3),
])
ds = datasets.ImageFolder('crop', transform=tf_)
n_val = int(len(ds) * 0.2)
tr, va = random_split(ds, [len(ds)-n_val, n_val], generator=torch.Generator().manual_seed(SEED))
tl = DataLoader(tr, batch_size=32, shuffle=True)
vl = DataLoader(va, batch_size=256)

def train_eval(name, epochs=25):
    torch.manual_seed(SEED); np.random.seed(SEED)
    m = build_model(name, num_classes=len(ds.classes)).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    lossf = torch.nn.CrossEntropyLoss()
    for _ in range(epochs):
        m.train()
        for x, y in tl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); lossf(m(x), y).backward(); opt.step()
    m.eval(); P=[]; Y=[]
    with torch.no_grad():
        for x, y in vl:
            P.append(m(x.to(device)).argmax(1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

pred_cnn, ytrue = train_eval('cnn')
pred_ann, _     = train_eval('ann')
print('CNN acc', (pred_cnn==ytrue).mean(), '| ANN acc', (pred_ann==ytrue).mean())

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = ds.classes
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for a, (title, pred) in zip(ax, [('ANN', pred_ann), ('CNN', pred_cnn)]):
    cm = confusion_matrix(ytrue, pred, normalize='true')
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=a, colorbar=False, values_format='.2f')
    a.set_title(f'{title}  (val acc {(pred==ytrue).mean():.3f})')
plt.tight_layout(); plt.show()

# 클래스별 정확도
print(f'{"code":>6} {"ANN":>7} {"CNN":>7}')
for i, c in enumerate(labels):
    mask = ytrue == i
    print(f'{c:>6} {(pred_ann[mask]==i).mean():>7.3f} {(pred_cnn[mask]==i).mean():>7.3f}')

In [ ]:
# CNN 은 맞추고 ANN 은 틀린 샘플 — 모양 구분에서 ANN 이 무너지는 지점
import torchvision
va_idx = va.indices
bad = [i for i in range(len(ytrue)) if pred_ann[i] != ytrue[i] and pred_cnn[i] == ytrue[i]][:16]
if bad:
    imgs = torch.stack([ds[va_idx[i]][0] for i in bad]) * 0.5 + 0.5
    grid = torchvision.utils.make_grid(imgs, nrow=8)
    plt.figure(figsize=(14, 4)); plt.imshow(grid.permute(1, 2, 0)); plt.axis('off')
    plt.title('ANN ❌  CNN ⭕  |  ' + ', '.join(f'{labels[ytrue[i]]}→ANN:{labels[pred_ann[i]]}' for i in bad[:6]))
    plt.show()
else:
    print('해당 샘플 없음')